# Nebula Gemma 7B QLoRA
This notebook trains **Gemma only** as Nebula's daily chat brain. It uses assistant-only loss, audits every trace, evaluates the base and adapter, refuses deployment if the quality gate fails, and exports Q4/Q5 GGUF files for LM Studio.

Before starting: request/accept access to `google/gemma-7b-it` on Hugging Face, add a private Colab secret named `HF_TOKEN`, select a GPU runtime, and upload the bundle created by `python training/scripts/build_colab_bundle.py`. Do not paste the token into a code cell.

In [ ]:
!pip -q install 'transformers>=4.48.0,<5.0' 'datasets>=3.3.0' 'peft>=0.14.0' 'accelerate>=1.4.0' 'bitsandbytes>=0.46.1' 'huggingface-hub>=0.28.0' 'safetensors>=0.5.0'


In [ ]:
import os, shutil, zipfile
from pathlib import Path
from google.colab import drive, files, userdata
from huggingface_hub import login

token = userdata.get('HF_TOKEN')
if not token:
    raise RuntimeError('Add HF_TOKEN in Colab Secrets before continuing.')
os.environ['HF_TOKEN'] = token
login(token=token, add_to_git_credential=False)
drive.mount('/content/drive')
os.chdir('/content')
uploaded = files.upload()
bundle = next((name for name in uploaded if name.endswith('.zip')), None)
if not bundle:
    raise RuntimeError('Upload nebula-gemma-colab-bundle.zip')
work = Path('/content/nebula-gemma')
shutil.rmtree(work, ignore_errors=True)
work.mkdir(parents=True)
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(work)
os.chdir(work)
print('Workspace:', work)


In [ ]:
!python training/scripts/check_environment.py


In [ ]:
import glob, subprocess, sys
inputs = sorted(glob.glob('training/data/synthetic-gemma-v1.jsonl') + glob.glob('training/exports/*.jsonl'))
command = [sys.executable, 'training/scripts/prepare_dataset.py', '--output-dir', 'training/data', '--validation-percent', '15']
for path in inputs:
    command.extend(['--input', path])
subprocess.run(command, check=True)
subprocess.run([sys.executable, 'training/scripts/validate_dataset.py', '--train', 'training/data/train.jsonl', '--validation', 'training/data/validation.jsonl', '--minimum-examples', '300', '--report', 'training/data/validation-report.json'], check=True)


In [ ]:
from pathlib import Path
drive_root = Path('/content/drive/MyDrive/NebulaTraining/nebula-gemma-7b-v2')
adapter_dir = drive_root / 'adapter'
reports_dir = drive_root / 'reports'
merged_dir = drive_root / 'merged-hf'
gguf_dir = drive_root / 'gguf'
for path in (adapter_dir, reports_dir, gguf_dir):
    path.mkdir(parents=True, exist_ok=True)
print('Artifacts will persist at', drive_root)


In [ ]:
!python training/scripts/train_qlora.py --config training/configs/gemma-7b-nebula-qlora.json --output-dir "{adapter_dir}" --resume


In [ ]:
!python training/scripts/evaluate_adapter.py --label base --output "{reports_dir / 'base-eval.json'}"
!python training/scripts/evaluate_adapter.py --label nebula-gemma-7b-v2 --adapter "{adapter_dir}" --output "{reports_dir / 'adapter-eval.json'}"
!python training/scripts/compare_eval_reports.py --baseline "{reports_dir / 'base-eval.json'}" --adapter "{reports_dir / 'adapter-eval.json'}" --output "{reports_dir / 'nebula-gemma-7b-v2-eval-gate.json'}"


The next cells should run only if the evaluation gate above passes. A failed gate is a real result: inspect the report, improve the dataset/config, and retrain instead of deploying a worse adapter.

In [ ]:
!python training/scripts/merge_adapter.py --adapter "{adapter_dir}" --output "{merged_dir}"


In [ ]:
import os, subprocess
os.chdir('/content')
if not Path('/content/llama.cpp').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git'], check=True)
subprocess.run(['cmake', '-S', '/content/llama.cpp', '-B', '/content/llama.cpp/build', '-DGGML_CUDA=ON'], check=True)
subprocess.run(['cmake', '--build', '/content/llama.cpp/build', '--config', 'Release', '-j', '2'], check=True)
f16 = gguf_dir / 'nebula-gemma-7b-v1-f16.gguf'
q4 = gguf_dir / 'nebula-gemma-7b-v1-Q4_K_M.gguf'
q5 = gguf_dir / 'nebula-gemma-7b-v1-Q5_K_M.gguf'
subprocess.run([sys.executable, '/content/llama.cpp/convert_hf_to_gguf.py', str(merged_dir), '--outfile', str(f16), '--outtype', 'f16'], check=True)
quantize = '/content/llama.cpp/build/bin/llama-quantize'
subprocess.run([quantize, str(f16), str(q4), 'Q4_K_M'], check=True)
subprocess.run([quantize, str(f16), str(q5), 'Q5_K_M'], check=True)
print('LM Studio builds:', q4, q5, sep='\n')


Import the Q4 build into LM Studio first. Keep the old Gemma model for rollback, run Nebula Bench, and assign the new model to Daily only after the local smoke tests pass. Qwen and the review model remain unchanged.